# Import dependencies

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
from scipy.signal import windows
from scipy.fft import fft, dct
import matplotlib.pyplot as plt
from IPython.display import Math, display
import sys

# Define variables and constants

In [ ]:
T = 1e-15                                               # Sampling period
Fs = 1/(T)                                              # Sampling frequency
L = 2501                                                # Length of correlation time vector
t = np.arange(0, L) * T                                 # Time vector
N_windows = 400                                         # Number of correlation windows

f = Fs * np.arange(0, (L//2)+1) / (L * 10**12) / 2      # Frequency vector with "dct" correction

oxid = 20                                               # Oxidation degree
thickness = 9                                           # Slab thickness

water_spect = []                                        # Initialization
graph_spect = []                                        # Initialization
V_water = []                                            # Initialization
V_graph = []                                            # Initialization

# Process VACF and compute phonon spectra

In [ ]:
path = '../../lammps/PDOS'

for i in range(1, N_windows+1):
    T1 = []
    T2 = []
    try:
    # Read data from file
        filename_water = path + f'/vacf_water_{thickness}.{i}'
        T1 = pd.read_table(filename_water, delimiter=' ', comment='#', header=None)
        acfw = T1.iloc[:, 1].values
        V_water.append(acfw[:L])

        filename_graph = path + f'/vacf_graph_{thickness}.{i}'
        T2 = pd.read_table(filename_graph, delimiter=' ', comment='#', header=None)
        acfg = T2.iloc[:, 1].values
        V_graph.append(acfg[:L])

    except Exception as e:
        print(f"Check N_windows variable: {e}")
        sys.exit(1)

V_water_avg = np.mean(V_water, axis=0)
V_graph_avg = np.mean(V_graph, axis=0)
V_water_avg = V_water_avg / V_water_avg[0]
V_graph_avg = V_graph_avg / V_graph_avg[0]


# Perform DCT
Y = dct(V_water_avg)
Y2 = dct(V_graph_avg)
P2 = np.abs(Y/L)
P1 = P2[:L//2+1]
P1[1:-1] = 2 * P1[1:-1]
P22 = np.abs(Y2/L)
P12 = P22[:L//2+1]
P12[1:-1] = 2 * P12[1:-1]
water_spect.append(P1)
graph_spect.append(P12)
w_spect_avg = np.mean(water_spect, axis=0)
g_spect_avg = np.mean(graph_spect, axis=0)

# Compute phonon overlap rate

In [ ]:
overlap = np.trapezoid(np.minimum(w_spect_avg,g_spect_avg)*f,f)
display(Math(rf"S = \int \omega f(\omega)\, d\omega"))
print(f'Phonon overlap rate S for {oxid}% oxidation:')
print(f"S = {overlap}")

# Plot phonon spectra

In [ ]:
fig = plt.figure(figsize=(2.22*1.5, 0.88*1.5), dpi=400)
grid = fig.add_gridspec(1,1, top=.99, right=.99, left=.01, bottom=.28)
ax = fig.add_subplot(grid[0])
ax.plot(f, w_spect_avg, '-', lw=1, color="#1F9BA2", label='Water')
ax.plot(f, g_spect_avg, '-', lw=1, color="#7E7E83", label=f'Graphene Oxide {oxid}%')
ax.set_yticklabels([])
ax.set_ylim([0,0.032])
ax.set_xlim([-3,140])
ax.fill_between(f,w_spect_avg*0, w_spect_avg,color="#1F9BA2", alpha=0.35)
ax.fill_between(f,g_spect_avg*0, g_spect_avg,color="#818181", alpha=0.45)
ax.set_ylabel('Phonon DOS (a.u)',)
ax.set_xlabel('Frequency (THz)')
ax.xaxis.set_major_formatter(plt.NullFormatter())
fig.set_layout_engine(pad=0.01, w_pad=0.01, h_pad=0.01)
plt.show()